In [60]:
from langgraph.graph import StateGraph,START,END 
from langchain_google_genai import GoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os


In [61]:

load_dotenv()

model = GoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [62]:
class BlogState(TypedDict):
    title: str
    outline:str
    content: str


In [66]:
#nodes

def outline_node(state:BlogState)->BlogState:
    """ this will give the outline of the topic"""
    title=state['title']

    prompt= f'generate a brief outline on the topic {title}'

    response = model.invoke(prompt)

    outline = response
    state['outline']= outline


    return state


def blog_node(state:BlogState)->BlogState:
    """ this will give the full blog on the topic """
    title=state['title']
    outline=state['outline']

    prompt= f'generate a brief blog on the topic {title} using this outline{outline}'

    response = model.invoke(prompt)

    blog = response
    state['blog']= blog

    return state





In [67]:
graph= StateGraph(BlogState)

#nodes

#outline
graph.add_node('outline_node',outline_node)


#blog
graph.add_node ('blog_node', blog_node)

#edges

graph.add_edge(START, 'outline_node')

graph.add_edge('outline_node', 'blog_node')

graph.add_edge('blog_node', END)


workflow = graph.compile()



In [70]:
initial_state= {"title":"make in india"}
resonse =workflow.invoke(initial_state)

print(resonse.outline)


AttributeError: 'dict' object has no attribute 'outline'